# Filosofi Data Ingestion (2014-2021)

Two-layer ingestion of INSEE FILOSOFI data (income and poverty) for Rhone department:
- **RAW**: Download source files as-is (ZIP files)
- **BRONZE**: Ingest with metadata (one CSV file per year)

Handles multiple source formats: XLS simple (2014-2016), CSV simple (2017-2018), CSV reconstructed (2019-2021)

In [1]:
print("Initialising dependencies...", flush=True)

import csv
import json
from io import BytesIO, TextIOWrapper
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
from zipfile import ZipFile
from datetime import datetime

import xlrd

print("Dependencies loaded.", flush=True)

Initialising dependencies...


Dependencies loaded.


In [2]:
# -- 1. PATH CONFIGURATION ---------------------------------------------------
# Resolve project root
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

BASE_DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = BASE_DATA_DIR / "raw" / "filosofi"
BRONZE_DIR = BASE_DATA_DIR / "bronze"

RAW_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

# Metadata
INGESTION_TIMESTAMP = datetime.now().isoformat()

print(f"Project root: {PROJECT_ROOT}")
print(f"RAW: {RAW_DIR}")
print(f"BRONZE: {BRONZE_DIR}")
print(f"Ingestion timestamp: {INGESTION_TIMESTAMP}")

Project root: /Users/zainfrayha/Desktop/electio-analytics-poc
RAW: /Users/zainfrayha/Desktop/electio-analytics-poc/data/raw/filosofi
BRONZE: /Users/zainfrayha/Desktop/electio-analytics-poc/data/bronze
Ingestion timestamp: 2026-04-01T19:29:55.428669


In [3]:
# -- 2. DATA CONFIGURATION ---------------------------------------------------
COMMUNES_URL = "https://geo.api.gouv.fr/departements/69/communes?fields=code&format=json&geometry=none"
HTTP_HEADERS = {"User-Agent": "Mozilla/5.0"}

UNAVAILABLE_YEARS = {
    2022: "Le millesime FILOSOFI 2022 n'a pas ete produit par l'Insee."
}

YEAR_SOURCES = {
    2014: {
        "mode": "xls_simple",
        "url": "https://www.insee.fr/fr/statistiques/fichier/3126432/filo-revenu-pauvrete-menage-2014.zip",
    },
    2015: {
        "mode": "xls_simple",
        "url": "https://www.insee.fr/fr/statistiques/fichier/3560121/filo-revenu-pauvrete-menage-2015.zip",
    },
    2016: {
        "mode": "xls_simple",
        "url": "https://www.insee.fr/fr/statistiques/fichier/4190004/filosofi-revenu-pauvrete-menage-2016.zip",
    },
    2017: {
        "mode": "csv_simple",
        "url": "https://www.insee.fr/fr/statistiques/fichier/4507225/base-filosofi-2017_CSV.zip",
    },
    2018: {
        "mode": "csv_simple",
        "url": "https://www.insee.fr/fr/statistiques/fichier/5009236/base-cc-filosofi-2018_CSV_geo2021.zip",
    },
    2019: {
        "mode": "csv_reconstructed",
        "url": "https://www.insee.fr/fr/statistiques/fichier/6036907/indic-struct-distrib-revenu-2019-COMMUNES_csv.zip",
    },
    2020: {
        "mode": "csv_reconstructed",
        "url": "https://www.insee.fr/fr/statistiques/fichier/6692220/indic-struct-distrib-revenu-2020-COMMUNES_csv.zip",
    },
    2021: {
        "mode": "csv_reconstructed",
        "url": "https://www.insee.fr/fr/statistiques/fichier/7756855/indic-struct-distrib-revenu-2021-COMMUNES_csv.zip",
    },
}

print(f"Years to process: {sorted(YEAR_SOURCES.keys())}")

Years to process: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]


In [4]:
# -- 3. UTILITY FUNCTIONS FOR DOWNLOADING AND PARSING -------------------------

def download_bytes(url: str) -> bytes:
    """Download file from URL and return bytes."""
    request = Request(url, headers=HTTP_HEADERS)
    with urlopen(request, timeout=60) as response:
        return response.read()


def fetch_rhone_codes() -> set[str]:
    """Fetch commune codes for Rhone department from geo.api.gouv.fr."""
    payload = download_bytes(COMMUNES_URL)
    communes = json.loads(payload.decode("utf-8"))
    return {item["code"] for item in communes if "code" in item}


def format_excel_value(value) -> str:
    """Format Excel cell value as string."""
    if value in ("", None):
        return ""
    if isinstance(value, float):
        if value.is_integer():
            return str(int(value))
        return format(value, ".15g")
    return str(value)


def normalize_code(value: str) -> str:
    """Normalize commune code to 5 digits."""
    value = value.strip()
    if value.isdigit():
        return value.zfill(5)
    return value


def rows_to_csv(path: Path, fieldnames: list[str], rows: list[dict[str, str]]) -> None:
    """Write rows to CSV file."""
    with path.open("w", newline="", encoding="utf-8") as output_file:
        writer = csv.DictWriter(output_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def choose_simple_csv_member(year: int, members: list[str]) -> str:
    """Find the appropriate CSV file for simple mode."""
    prefix = f"cc_filosofi_{year}_com"
    for member in members:
        lower_member = member.lower()
        if lower_member.startswith("meta_"):
            continue
        if prefix in lower_member and lower_member.endswith(".csv"):
            return member
    raise RuntimeError(f"No simple commune CSV found for {year}")


def choose_detailed_member(year: int, members: list[str], family: str) -> str:
    """Find the appropriate detailed CSV file for reconstructed mode."""
    family = family.lower()
    for member in members:
        lower_member = member.lower()
        if not lower_member.endswith(".csv"):
            continue
        if lower_member.startswith("meta_"):
            continue
        if f"filo{year}" in lower_member and family in lower_member and lower_member.endswith("_com.csv"):
            return member
    raise RuntimeError(f"No {family} commune CSV found for {year}")


print("Utility functions defined.")

Utility functions defined.


In [5]:
# -- 4. DATA LOADING FUNCTIONS -----------------------------------------------

def load_simple_xls_rows(year: int, zip_content: bytes) -> tuple[list[str], list[dict[str, str]]]:
    """Load data from XLS files (2014-2016)."""
    with ZipFile(BytesIO(zip_content)) as archive:
        member = next(name for name in archive.namelist() if name.lower().endswith(".xls"))
        workbook = xlrd.open_workbook(file_contents=archive.read(member))

    sheet = workbook.sheet_by_name("COM")
    fieldnames = [str(value) for value in sheet.row_values(5)]
    rows: list[dict[str, str]] = []

    for row_idx in range(6, sheet.nrows):
        row_values = sheet.row_values(row_idx)
        if not row_values:
            continue

        code = format_excel_value(row_values[0])
        if not code:
            continue

        row = {
            fieldname: format_excel_value(row_values[col_idx]) if col_idx < len(row_values) else ""
            for col_idx, fieldname in enumerate(fieldnames)
        }
        row["CODGEO"] = normalize_code(row["CODGEO"])
        rows.append(row)

    return fieldnames, rows


def load_simple_csv_rows(year: int, zip_content: bytes) -> tuple[list[str], list[dict[str, str]]]:
    """Load data from simple CSV files (2017-2018)."""
    with ZipFile(BytesIO(zip_content)) as archive:
        member = choose_simple_csv_member(year, archive.namelist())
        print(f"Reading {member}...", flush=True)
        with archive.open(member) as csv_file:
            reader = csv.DictReader(TextIOWrapper(csv_file, encoding="utf-8", newline=""), delimiter=";")
            rows = list(reader)
            fieldnames = list(reader.fieldnames or [])

    for row in rows:
        row["CODGEO"] = normalize_code(row["CODGEO"])

    return fieldnames, rows


def load_reconstructed_rows(year: int, zip_content: bytes) -> tuple[list[str], list[dict[str, str]]]:
    """Load data from reconstructed CSV files (2019-2021)."""
    suffix = str(year)[-2:]
    output_fields = [
        "CODGEO",
        f"NBMENFISC{suffix}",
        f"NBPERSMENFISC{suffix}",
        f"MED{suffix}",
        f"PIMP{suffix}",
        f"TP60{suffix}",
        f"TP60AGE1{suffix}",
        f"TP60AGE2{suffix}",
        f"TP60AGE3{suffix}",
        f"TP60AGE4{suffix}",
        f"TP60AGE5{suffix}",
        f"TP60AGE6{suffix}",
        f"TP60TOL1{suffix}",
        f"TP60TOL2{suffix}",
        f"PACT{suffix}",
        f"PTSA{suffix}",
        f"PCHO{suffix}",
        f"PBEN{suffix}",
        f"PPEN{suffix}",
        f"PPAT{suffix}",
        f"PPSOC{suffix}",
        f"PPFAM{suffix}",
        f"PPMINI{suffix}",
        f"PPLOGT{suffix}",
        f"PIMPOT{suffix}",
        f"D1{suffix}",
        f"D9{suffix}",
        f"RD{suffix}",
    ]

    with ZipFile(BytesIO(zip_content)) as archive:
        members = archive.namelist()
        dec_member = choose_detailed_member(year, members, "_dec_")
        disp_member = choose_detailed_member(year, members, "_disp_")
        pauvres_member = choose_detailed_member(year, members, "_disp_pauvres_")

        print(f"Reading {dec_member}...", flush=True)
        dec_rows = csv.DictReader(
            TextIOWrapper(archive.open(dec_member), encoding="utf-8", newline=""),
            delimiter=";",
        )
        print(f"Reading {disp_member}...", flush=True)
        disp_rows = csv.DictReader(
            TextIOWrapper(archive.open(disp_member), encoding="utf-8", newline=""),
            delimiter=";",
        )
        print(f"Reading {pauvres_member}...", flush=True)
        pauvres_rows = csv.DictReader(
            TextIOWrapper(archive.open(pauvres_member), encoding="utf-8", newline=""),
            delimiter=";",
        )

        merged: dict[str, dict[str, str]] = {}

        for row in disp_rows:
            code = normalize_code(row["CODGEO"])
            merged[code] = {
                "CODGEO": code,
                f"NBMENFISC{suffix}": row.get(f"NBMEN{suffix}", ""),
                f"NBPERSMENFISC{suffix}": row.get(f"NBPERS{suffix}", ""),
                f"MED{suffix}": row.get(f"Q2{suffix}", ""),
                f"PACT{suffix}": row.get(f"PACT{suffix}", ""),
                f"PTSA{suffix}": row.get(f"PTSA{suffix}", ""),
                f"PCHO{suffix}": row.get(f"PCHO{suffix}", ""),
                f"PBEN{suffix}": row.get(f"PBEN{suffix}", ""),
                f"PPEN{suffix}": row.get(f"PPEN{suffix}", ""),
                f"PPAT{suffix}": row.get(f"PPAT{suffix}", ""),
                f"PPSOC{suffix}": row.get(f"PPSOC{suffix}", ""),
                f"PPFAM{suffix}": row.get(f"PPFAM{suffix}", ""),
                f"PPMINI{suffix}": row.get(f"PPMINI{suffix}", ""),
                f"PPLOGT{suffix}": row.get(f"PPLOGT{suffix}", ""),
                f"PIMPOT{suffix}": row.get(f"PIMPOT{suffix}", ""),
                f"D1{suffix}": row.get(f"D1{suffix}", ""),
                f"D9{suffix}": row.get(f"D9{suffix}", ""),
                f"RD{suffix}": row.get("RD", ""),
            }

        for row in dec_rows:
            code = normalize_code(row["CODGEO"])
            if code not in merged:
                merged[code] = {"CODGEO": code}
            merged[code][f"PIMP{suffix}"] = row.get(f"PMIMP{suffix}", "")

        for row in pauvres_rows:
            code = normalize_code(row["CODGEO"])
            if code not in merged:
                merged[code] = {"CODGEO": code}
            merged[code].update(
                {
                    f"TP60{suffix}": row.get(f"TP60{suffix}", ""),
                    f"TP60AGE1{suffix}": row.get(f"AGE1TP60{suffix}", ""),
                    f"TP60AGE2{suffix}": row.get(f"AGE2TP60{suffix}", ""),
                    f"TP60AGE3{suffix}": row.get(f"AGE3TP60{suffix}", ""),
                    f"TP60AGE4{suffix}": row.get(f"AGE4TP60{suffix}", ""),
                    f"TP60AGE5{suffix}": row.get(f"AGE5TP60{suffix}", ""),
                    f"TP60AGE6{suffix}": row.get(f"AGE6TP60{suffix}", ""),
                    f"TP60TOL1{suffix}": row.get(f"TOL1TP60{suffix}", ""),
                    f"TP60TOL2{suffix}": row.get(f"TOL2TP60{suffix}", ""),
                }
            )

    rows = [{field: row.get(field, "") for field in output_fields} for row in merged.values()]
    return output_fields, rows


print("Data loading functions defined.")

Data loading functions defined.


In [6]:
# -- 5. FILTERING UTILITIES ---------------------------------------------------

def filter_rows(rows: list[dict[str, str]], codes_rhone: set[str]) -> list[dict[str, str]]:
    """Filter rows to only include Rhone communes."""
    if codes_rhone:
        return [row for row in rows if normalize_code(row["CODGEO"]) in codes_rhone]
    return [row for row in rows if normalize_code(row["CODGEO"]).startswith("69")]


print("Filtering functions defined.")

Filtering functions defined.


In [7]:
# -- 6. RAW LAYER: Download source files as-is ---------------------------
print("\n" + "="*70)
print("RAW LAYER: Downloading source files")
print("="*70)

raw_files = {}

for current_year in range(2014, 2023):
    if current_year in UNAVAILABLE_YEARS:
        print(f"\n⏭️  {current_year}: {UNAVAILABLE_YEARS[current_year]}")
        continue

    try:
        config = YEAR_SOURCES[current_year]
        # Store raw zip file with clear naming
        raw_path = RAW_DIR / f"filosofi_{current_year}_insee.zip"
        
        if raw_path.exists():
            print(f"✅ {current_year}: Already in RAW - {raw_path.name}")
        else:
            print(f"\n📥 {current_year}: Downloading...")
            zip_content = download_bytes(config["url"])
            with open(raw_path, "wb") as f:
                f.write(zip_content)
            print(f"✅ {current_year}: Saved to RAW - {raw_path.name}")
        
        raw_files[current_year] = raw_path
        
    except Exception as exc:
        print(f"❌ {current_year}: Failed - {type(exc).__name__}: {exc}")

print(f"\n✅ RAW layer complete. Files: {len(raw_files)}")


RAW LAYER: Downloading source files
✅ 2014: Already in RAW - filosofi_2014_insee.zip
✅ 2015: Already in RAW - filosofi_2015_insee.zip
✅ 2016: Already in RAW - filosofi_2016_insee.zip
✅ 2017: Already in RAW - filosofi_2017_insee.zip
✅ 2018: Already in RAW - filosofi_2018_insee.zip
✅ 2019: Already in RAW - filosofi_2019_insee.zip
✅ 2020: Already in RAW - filosofi_2020_insee.zip
✅ 2021: Already in RAW - filosofi_2021_insee.zip

⏭️  2022: Le millesime FILOSOFI 2022 n'a pas ete produit par l'Insee.

✅ RAW layer complete. Files: 8


In [8]:
# -- 7. Fetch Rhone codes for filtering --------------------------------
print("\nFetching Rhone communes for filtering...")

try:
    rhone_codes = fetch_rhone_codes()
    print(f"✅ Rhone communes fetched: {len(rhone_codes)}")
except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as exc:
    print(f"⚠️  WARNING: Could not fetch Rhone communes ({exc}). Using fallback '69' prefix.")
    rhone_codes = set()


Fetching Rhone communes for filtering...


✅ Rhone communes fetched: 266


In [9]:
# -- 8. BRONZE LAYER: Ingest from RAW with metadata ----------------------
print("\n" + "="*70)
print("BRONZE LAYER: Ingestion with metadata (per year)")
print("="*70)

bronze_files = {}

for current_year in range(2014, 2023):
    if current_year not in raw_files:
        continue

    try:
        config = YEAR_SOURCES[current_year]
        raw_path = raw_files[current_year]
        bronze_path = BRONZE_DIR / f"filosofi_{current_year}_rhone_69_bronze.csv"
        
        print(f"\n📦 {current_year}: Processing RAW -> BRONZE")
        
        # Read from raw zip file
        with open(raw_path, "rb") as f:
            zip_content = f.read()

        # Load based on mode
        if config["mode"] == "xls_simple":
            fieldnames, rows = load_simple_xls_rows(current_year, zip_content)
        elif config["mode"] == "csv_simple":
            fieldnames, rows = load_simple_csv_rows(current_year, zip_content)
        elif config["mode"] == "csv_reconstructed":
            fieldnames, rows = load_reconstructed_rows(current_year, zip_content)
        else:
            raise RuntimeError(f"Unsupported mode for {current_year}: {config['mode']}")

        # Filter to Rhone
        rhone_rows = filter_rows(rows, rhone_codes)
        
        # Add metadata columns
        metadata_cols = ["year", "extraction_source_url", "ingestion_timestamp", "source_file_name"]
        
        for row in rhone_rows:
            row["year"] = str(current_year)
            row["extraction_source_url"] = config["url"]
            row["ingestion_timestamp"] = INGESTION_TIMESTAMP
            row["source_file_name"] = f"INSEE_FILOSOFI_{current_year}"
        
        # Add metadata columns to fieldnames if not already there
        for col in metadata_cols:
            if col not in fieldnames:
                fieldnames.append(col)
        
        # Save to bronze
        rows_to_csv(bronze_path, fieldnames, rhone_rows)
        bronze_files[current_year] = bronze_path
        
        print(f"   ✅ Rhone communes: {len(rhone_rows)}")
        if rhone_codes and len(rhone_rows) != len(rhone_codes):
            print(f"   ⚠️  Expected {len(rhone_codes)}, found {len(rhone_rows)}")
        print(f"   ✅ Saved: {bronze_path.name}")

    except Exception as exc:
        print(f"❌ {current_year}: Failed - {type(exc).__name__}: {exc}")

print(f"\n✅ BRONZE layer complete. Files: {len(bronze_files)}")


BRONZE LAYER: Ingestion with metadata (per year)

📦 2014: Processing RAW -> BRONZE


   ✅ Rhone communes: 266
   ✅ Saved: filosofi_2014_rhone_69_bronze.csv

📦 2015: Processing RAW -> BRONZE


   ✅ Rhone communes: 266
   ✅ Saved: filosofi_2015_rhone_69_bronze.csv

📦 2016: Processing RAW -> BRONZE


   ✅ Rhone communes: 265
   ⚠️  Expected 266, found 265
   ✅ Saved: filosofi_2016_rhone_69_bronze.csv

📦 2017: Processing RAW -> BRONZE
Reading cc_filosofi_2017_COM.CSV...


   ✅ Rhone communes: 265
   ⚠️  Expected 266, found 265
   ✅ Saved: filosofi_2017_rhone_69_bronze.csv

📦 2018: Processing RAW -> BRONZE
Reading cc_filosofi_2018_COM-geo2021.CSV...


...

Reading FILO2020_DISP_PAUVRES_COM.csv...


   ✅ Rhone communes: 265
   ⚠️  Expected 266, found 265
   ✅ Saved: filosofi_2020_rhone_69_bronze.csv

📦 2021: Processing RAW -> BRONZE
Reading FILO2021_DEC_COM.csv...


Reading FILO2021_DISP_COM.csv...


Reading FILO2021_DISP_PAUVRES_COM.csv...


   ✅ Rhone communes: 265
   ⚠️  Expected 266, found 265
   ✅ Saved: filosofi_2021_rhone_69_bronze.csv

✅ BRONZE layer complete. Files: 8


In [10]:
# -- 9. INGESTION SUMMARY ---------------------------------------------------
print("\n" + "="*70)
print("FILOSOFI INGESTION COMPLETE")
print("="*70)

print(f"\nTimestamp: {INGESTION_TIMESTAMP}")
print(f"\n📁 RAW Layer ({RAW_DIR}):")
for year, path in sorted(raw_files.items()):
    print(f"   ✅ {path.name}")

print(f"\n📁 BRONZE Layer ({BRONZE_DIR}):")
for year, path in sorted(bronze_files.items()):
    print(f"   ✅ {path.name}")

print(f"\n📊 Summary:")
print(f"   Total years processed: {len(bronze_files)}")
print(f"   Years: {sorted(bronze_files.keys())}")

print(f"\n✅ Next steps: Run transformation notebook to merge and process SILVER layer")
print("="*70)


FILOSOFI INGESTION COMPLETE

Timestamp: 2026-04-01T19:29:55.428669

📁 RAW Layer (/Users/zainfrayha/Desktop/electio-analytics-poc/data/raw/filosofi):
   ✅ filosofi_2014_insee.zip
   ✅ filosofi_2015_insee.zip
   ✅ filosofi_2016_insee.zip
   ✅ filosofi_2017_insee.zip
   ✅ filosofi_2018_insee.zip
   ✅ filosofi_2019_insee.zip
   ✅ filosofi_2020_insee.zip
   ✅ filosofi_2021_insee.zip

📁 BRONZE Layer (/Users/zainfrayha/Desktop/electio-analytics-poc/data/bronze):
   ✅ filosofi_2014_rhone_69_bronze.csv
   ✅ filosofi_2015_rhone_69_bronze.csv
   ✅ filosofi_2016_rhone_69_bronze.csv
   ✅ filosofi_2017_rhone_69_bronze.csv
   ✅ filosofi_2018_rhone_69_bronze.csv
   ✅ filosofi_2019_rhone_69_bronze.csv
   ✅ filosofi_2020_rhone_69_bronze.csv
   ✅ filosofi_2021_rhone_69_bronze.csv

📊 Summary:
   Total years processed: 8
   Years: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]

✅ Next steps: Run transformation notebook to merge and process SILVER layer
